In [ ]:
# --- Cell 1: Standard Imports ---
import sys
import json
import time
import re
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
import ipywidgets as widgets
from IPython.display import display, clear_output

print(f"✅ Environment Ready (Running on {sys.platform})")

In [ ]:
# --- Cell 2: Submission Engine ---

# ⚠️ HARDCODE YOUR QUEUE PATH HERE ⚠️
QUEUE_DIR = Path("/home/SDSMT.LOCAL/bscott/petakit_jobs/queue") 

def parse_z_step(acq_file, default=1.0):
    if not acq_file.exists(): return default
    try:
        with open(acq_file, 'r', errors='ignore') as f:
            content = f.read()
        match = re.search(r"Z\s*Step.*?(\d+\.?\d*)", content, re.IGNORECASE)
        if match: return float(match.group(1))
    except: pass
    return default

def roi_to_str(slice_tuple):
    if slice_tuple is None: return "0:10,0:10"
    y, x = slice_tuple
    return f"{y.start}:{y.stop},{x.start}:{x.stop}"

def submit_crop_job(base_file, top_roi, bot_roi, channels, rotate=True):
    job_id = f"CROP_{base_file.stem}_{int(time.time()*1000)}"
    json_path = QUEUE_DIR / f"{job_id}.json"
    
    payload = {
        "jobType": "OPM_CROP",
        "inputPath": str(base_file),
        "parameters": {
            "rois": {"top": roi_to_str(top_roi), "bottom": roi_to_str(bot_roi)},
            "channels": channels, # List of ints [0, 1, 2, 3]
            "rotate": rotate,
            "outputFormat": "TIFF_SERIES"
        },
        "submittedBy": "Student_Client"
    }
    with open(json_path, 'w') as f: json.dump(payload, f, indent=4)
    return json_path

def submit_deskew_job(input_dir, z_step, angle=31.8, pixel=0.136):
    job_id = f"DESKEW_{input_dir.name}_{int(time.time()*1000)}"
    json_path = QUEUE_DIR / f"{job_id}.json"
    
    payload = {
        "jobType": "DESKEW_ROTATE",
        "inputPath": str(input_dir),
        "parameters": {
            "z_step_um": z_step,
            "sheet_angle": angle,
            "pixel_size_um": pixel,
            "deskew": True, "rotate": True
        }
    }
    with open(json_path, 'w') as f: json.dump(payload, f, indent=4)
    return json_path

In [ ]:
# --- Cell 3: Select File & Draw ROIs ---

# Global storage for the next cell to access
CURRENT_FILE = None
DETECTED_CHANNELS = 4 # Default
ROIS = {"top": None, "bot": None}

def on_top(eclick, erelease):
    ext = selector_top.extents 
    ROIS["top"] = (slice(int(ext[2]), int(ext[3])), slice(int(ext[0]), int(ext[1])))
    selector_top.set_active(False); selector_bot.set_active(True)
    lbl_status.value = "✅ Top set. Now draw BLUE box for Bottom ROI."

def on_bot(eclick, erelease):
    ext = selector_bot.extents
    ROIS["bot"] = (slice(int(ext[2]), int(ext[3])), slice(int(ext[0]), int(ext[1])))
    lbl_status.value = "✅ Both ROIs set! Scroll down to Channel Selection."

# UI Layout
path_input = widgets.Text(placeholder="/path/to/file_MMStack_Pos0.ome.tif", description="File:", layout=widgets.Layout(width='80%'))
btn_load = widgets.Button(description="Load File", icon="folder-open")
output_viz = widgets.Output()
lbl_status = widgets.Label("Waiting for file...")

def on_load_click(b):
    global CURRENT_FILE, DETECTED_CHANNELS, selector_top, selector_bot
    p = Path(path_input.value)
    if not p.exists(): lbl_status.value = "❌ File not found!"; return
    CURRENT_FILE = p
    
    with output_viz:
        clear_output()
        print("Loading preview...")
        with tifffile.TiffFile(p) as tif:
            # Simple shape heuristic for OPM data
            if len(tif.series[0].shape) == 5: # T, Z, C, Y, X
                DETECTED_CHANNELS = tif.series[0].shape[2]
                mip = np.max(tif.series[0].asarray()[0, :, 0, :, :], axis=0)
            elif len(tif.series[0].shape) == 4: # Z, C, Y, X or T, C, Y, X
                # Guess C is dim 1
                DETECTED_CHANNELS = tif.series[0].shape[1] 
                mip = np.max(tif.series[0].asarray()[0, 0, :, :], axis=0) # Very rough preview
            else:
                mip = tif.series[0].asarray()[0] # Fallback
                
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(mip, cmap='gray', vmax=np.percentile(mip, 99))
        ax.set_title(f"Draw RED (Top) -> BLUE (Bottom)\nDetected {DETECTED_CHANNELS} Input Channels")
        
        selector_top = RectangleSelector(ax, on_top, useblit=True, props=dict(facecolor='red', alpha=0.3))
        selector_bot = RectangleSelector(ax, on_bot, useblit=True, props=dict(facecolor='blue', alpha=0.3), interactive=False)
        plt.show()
        lbl_status.value = "Draw the RED box (Top Camera) first."

btn_load.on_click(on_load_click)
display(widgets.VBox([widgets.HBox([path_input, btn_load]), lbl_status]), output_viz)

In [ ]:
# --- Cell 4: Channel Selection & Save Template ---

# 1. Generate Channel Checkboxes
print(f"Creating widgets for {DETECTED_CHANNELS*2} output channels...")
channel_widgets = []
ui_grid = []

# Create a grid of checkboxes (Input Ch -> Output Ch A/B)
# Assumption: 1 Input Channel = 2 Output Channels (Top/Bot)
for i in range(DETECTED_CHANNELS):
    out_a = i * 2
    out_b = i * 2 + 1
    
    lbl = widgets.Label(f"Input Ch {i}:")
    cb_a = widgets.Checkbox(value=True, description=f"Out {out_a}")
    cb_b = widgets.Checkbox(value=True, description=f"Out {out_b}")
    
    channel_widgets.append((out_a, cb_a))
    channel_widgets.append((out_b, cb_b))
    ui_grid.append(widgets.HBox([lbl, cb_a, cb_b]))

# 2. Other Settings
w_angle = widgets.FloatText(value=31.8, description="Angle:")
w_pix = widgets.FloatText(value=0.136, description="Pixel (um):")
w_rot = widgets.Checkbox(value=True, description="Rotate 90°")
btn_save = widgets.Button(description="Submit & Save Settings", button_style='success', icon='save')
out_log = widgets.Output()

display(widgets.VBox([
    widgets.Label("<b>Select Output Channels to Keep:</b>"),
    widgets.VBox(ui_grid),
    widgets.HTML("<hr>"),
    widgets.HBox([w_angle, w_pix, w_rot]),
    btn_save,
    out_log
]))

TEMPLATE_FILE = None

def on_save(b):
    global TEMPLATE_FILE
    with out_log:
        out_log.clear_output()
        if ROIS["top"] is None: print("❌ ROIs not set in Cell 3!"); return
        if CURRENT_FILE is None: print("❌ No file loaded!"); return
        
        # Gather Channels
        selected_channels = [ch_id for ch_id, w in channel_widgets if w.value]
        print(f"✅ Selected Channels: {selected_channels}")
        
        # 1. Submit Single Job (Test)
        print(f"🚀 Submitting Crop for current file...")
        submit_crop_job(CURRENT_FILE, ROIS["top"], ROIS["bot"], selected_channels, w_rot.value)
        
        # 2. Make Dir
        base_name = CURRENT_FILE.name
        if base_name.lower().endswith(".ome.tif"): clean_name = base_name[:-8]
        elif base_name.lower().endswith(".tif"): clean_name = base_name[:-4]
        else: clean_name = CURRENT_FILE.stem
        target_dir = CURRENT_FILE.parent / clean_name
        if not target_dir.exists(): target_dir.mkdir(parents=True)
        
        # 3. Submit Deskew
        meta_file = CURRENT_FILE.parent / "AcqSettings.txt"
        z_val = parse_z_step(meta_file, 1.0)
        submit_deskew_job(target_dir, z_val, w_angle.value, w_pix.value)
        
        # 4. Save JSON
        template_data = {
            "source_file": CURRENT_FILE.name,
            "channels": selected_channels,
            "rotate": w_rot.value,
            "rois": {"top": roi_to_str(ROIS["top"]), "bottom": roi_to_str(ROIS["bot"])},
            "deskew": {"angle": w_angle.value, "pixel": w_pix.value, "default_z": 1.0}
        }
        
        TEMPLATE_FILE = target_dir / "processing_settings.json"
        with open(TEMPLATE_FILE, "w") as f: json.dump(template_data, f, indent=4)
        print(f"💾 Template saved to: {TEMPLATE_FILE.name}")
        print("👇 You can now run Cell 5 to process the rest.")

btn_save.on_click(on_save)

In [ ]:
# --- Cell 5: Batch Process Folder ---

if TEMPLATE_FILE is None:
    print("⚠️ Please run Cell 4 first to generate the settings template.")
else:
    with open(TEMPLATE_FILE, 'r') as f:
        settings = json.load(f)
        
    # Helper to parse ROI string back to slices
    def str_to_roi(s):
        y, x = s.split(',')
        return (slice(*map(int, y.split(':'))), slice(*map(int, x.split(':'))))

    top_roi = str_to_roi(settings["rois"]["top"])
    bot_roi = str_to_roi(settings["rois"]["bottom"])
    
    # Scan root (Parent of Parent of template file)
    scan_root = TEMPLATE_FILE.parent.parent
    
    print(f"📂 Scanning: {scan_root}")
    all_files = sorted(list(scan_root.glob("**/*_MMStack_Pos0.ome.tif")))
    files_to_run = [f for f in all_files if f.name != settings["source_file"]]
    
    print(f"🚀 Found {len(files_to_run)} other files. Submitting tickets...")
    
    for i, f in enumerate(files_to_run):
        print(f"   [{i+1}/{len(files_to_run)}] {f.name}")
        
        # 1. Crop
        submit_crop_job(f, top_roi, bot_roi, settings["channels"], settings["rotate"])
        
        # 2. Create Dir
        base_name = f.name
        if base_name.lower().endswith(".ome.tif"): clean_name = base_name[:-8]
        elif base_name.lower().endswith(".tif"): clean_name = base_name[:-4]
        else: clean_name = f.stem
        target_dir = f.parent / clean_name
        if not target_dir.exists(): target_dir.mkdir(parents=True)
        
        # 3. Deskew
        meta_file = f.parent / "AcqSettings.txt"
        z_val = parse_z_step(meta_file, settings["deskew"]["default_z"])
        submit_deskew_job(target_dir, z_val, settings["deskew"]["angle"], settings["deskew"]["pixel"])
        
    print("\n✅ All batch jobs submitted!")